# Meal Logging Agent — Evaluation Notebook

Test each component of the agent in isolation, then run full round-trips.
Use this to iterate on prompt quality before changing production code.

**Run order:** cells are independent except Cell 5 (needs Cell 4 results).

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path
from dotenv import load_dotenv

# Load credentials from .env
load_dotenv(Path(os.getcwd()).parent / ".env")

# Add src/ to path so backend imports work
sys.path.insert(0, str(Path(os.getcwd()).parent / "src"))

print("GOOGLE_API_KEY set:", bool(os.environ.get("GOOGLE_API_KEY")))
print("USDA_API_KEY    set:", bool(os.environ.get("USDA_API_KEY")))
print("DATABASE_URL    set:", bool(os.environ.get("DATABASE_URL")))

---
## Cell 1 — Entity Extraction
Tests `extract_entities()` across varied inputs.  
**Pass criteria:** vague quantities (`"a handful"`, `"some"`) → `confidence < 0.6`

In [ ]:
from backend.agent.tools.meals import extract_entities

test_inputs = [
    "2 scrambled eggs and a slice of whole wheat toast",
    "a handful of mixed nuts",                   # vague → confidence < 0.6
    "big bowl of pasta with tomato sauce",        # vague portion
    "200g grilled chicken breast",               # explicit weight
    "coffee with a splash of oat milk and sugar",# multiple vague items
    "protein shake: 1 scoop whey, 250ml whole milk, 1 banana",
]

for inp in test_inputs:
    print(f"INPUT: {inp}")
    result = extract_entities(inp)
    for e in result.entities:
        flag = " ⚠ LOW CONF" if e.confidence < 0.6 else ""
        print(f"  {e.food_name:40s}  {e.qty_g:6.0f}g  ({e.unit})  conf={e.confidence:.2f}{flag}")
    print()

---
## Cell 2 — Image Classification
Tests `classify_food_image()` — set `img_path` to a local food image.  
**Pass criteria:** `is_food=False` on non-food images.

In [ ]:
import base64
from backend.agent.tools.meals import classify_food_image

img_path = "path/to/your/food_image.jpg"  # ← change this

if not os.path.exists(img_path):
    print("No image found at", img_path)
    print("Set img_path to a local .jpg or .png file to test image classification.")
else:
    with open(img_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()

    result = classify_food_image(b64)
    print("is_food   :", result.is_food)
    print("food_items:", result.food_items)
    print("ingredients:", result.ingredients)
    if result.reason:
        print("reason    :", result.reason)

---
## Cell 3 — Correction Loop
Tests `apply_correction()` for quantity changes, additions, removals, and substitutions.  
**Pass criteria:** entity list matches intent for all four correction types.

In [ ]:
from backend.agent.tools.meals import FoodEntity, apply_correction

base_entities = [
    FoodEntity(food_name="scrambled egg", qty_g=122.0, unit="2 eggs").model_dump(),
    FoodEntity(food_name="whole wheat bread", qty_g=28.0, unit="slice").model_dump(),
]

corrections = [
    ("actually 3 eggs",             "qty change"),
    ("add a black coffee",           "addition"),
    ("remove the toast",             "removal"),
    ("almond milk not oat milk",     "substitution"),
    ("3 eggs and add orange juice",  "multiple"),
]

for correction, label in corrections:
    result = apply_correction(base_entities, correction)
    print(f"[{label}] '{correction}'")
    for e in result.entities:
        print(f"  {e.food_name:40s}  {e.qty_g:.0f}g")
    print()

---
## Cell 4 — USDA Batch Lookup
Tests `usda_lookup_batch()` — cache-first, FDC API on miss.  
**Pass criteria:** nutrients scaled to `qty_g`, `fdc_id != -1` for known foods.

In [ ]:
import asyncio
from backend.agent.tools.meals import FoodEntity
from backend.agent.tools.usda import usda_lookup_batch

entities = [
    FoodEntity(food_name="scrambled egg",      qty_g=122.0, unit="2 eggs"),
    FoodEntity(food_name="whole wheat toast",  qty_g=28.0,  unit="slice"),
    FoodEntity(food_name="banana",             qty_g=118.0, unit="medium"),
    FoodEntity(food_name="xyzzy_fake_food",    qty_g=100.0, unit="g"),   # should miss
]

results = await usda_lookup_batch(entities)

for r in results:
    status = "✅" if r.fdc_id != -1 else "❌ MISS"
    print(f"{status}  {r.food_name[:45]:45s}  conf={r.match_confidence:.2f}")
    if r.fdc_id != -1:
        print(f"       {r.kcal:.0f} kcal  |  {r.protein_g:.1f}g protein  |  "
              f"{r.carbs_g:.1f}g carbs  |  {r.fat_g:.1f}g fat  |  {r.fiber_g:.1f}g fiber")
        print(f"       {len(r.nutrients)} total nutrients")
    print()

---
## Cell 5 — Confirmation Card Formatter
Tests the pure-Python `format_confirmation_card()`.  
Requires Cell 4 to have run (uses `results`).

In [ ]:
from backend.agent.tools.meals import format_confirmation_card

card = format_confirmation_card(results)  # results from Cell 4
print(card)

---
## Cell 6 — Full Text Round-Trip (no DB)
Runs extract → USDA batch → format card in one shot, without touching the graph or DB.  
**Pass criteria:** card is correct, totals make sense.

In [ ]:
from backend.agent.tools.meals import extract_entities, format_confirmation_card
from backend.agent.tools.usda import usda_lookup_batch

meal_text = "had 2 scrambled eggs, a slice of whole wheat toast, and black coffee"

extracted = extract_entities(meal_text)
print("Extracted entities:")
for e in extracted.entities:
    print(f"  {e.food_name:40s}  {e.qty_g:.0f}g  conf={e.confidence:.2f}")
print()

usda_results = await usda_lookup_batch(extracted.entities)
card = format_confirmation_card(usda_results)
print("Confirmation card:")
print(card)

---
## Cell 7 — Full Graph Round-Trip (live DB + Redis)
Runs the full LangGraph agent: extract → USDA → present → confirm → DB write.  
Requires Postgres and Redis to be running.

**Pass criteria:** card returned on first call (`paused=True`), meal logged on second call (`paused=False`).

In [ ]:
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from backend.agent.orchestrator import handle_message

DB_URL = os.environ["DATABASE_URL"].replace("postgresql+asyncpg://", "postgresql://")
USER_ID = "00000000-0000-0000-0000-000000000001"
THREAD_ID = "notebook-test-thread"

async with AsyncPostgresSaver.from_conn_string(DB_URL) as checkpointer:
    await checkpointer.setup()

    # Step 1: send meal
    r1 = await handle_message(
        user_id=USER_ID,
        user_input="2 scrambled eggs and whole wheat toast",
        thread_id=THREAD_ID,
        checkpointer=checkpointer,
    )
    print("── Step 1: Confirmation card ──")
    print(r1["message"])
    print(f"\npaused={r1['paused']}  (expected True)")
    print()

    # Step 2: confirm
    r2 = await handle_message(
        user_id=USER_ID,
        user_input="yes log it",
        thread_id=THREAD_ID,
        checkpointer=checkpointer,
    )
    print("── Step 2: Logged ──")
    print(r2["message"])
    print(f"\npaused={r2['paused']}  (expected False)")

---
## Cell 8 — Correction Loop Round-Trip (live)
Sends a meal, applies a correction, then confirms.

**Pass criteria:** nutrient totals change to reflect the correction.

In [ ]:
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from backend.agent.orchestrator import handle_message

DB_URL = os.environ["DATABASE_URL"].replace("postgresql+asyncpg://", "postgresql://")
USER_ID = "00000000-0000-0000-0000-000000000001"
THREAD_ID = "notebook-correction-thread"

async with AsyncPostgresSaver.from_conn_string(DB_URL) as checkpointer:
    await checkpointer.setup()

    r1 = await handle_message(
        user_id=USER_ID,
        user_input="a banana and a cup of oat milk",
        thread_id=THREAD_ID,
        checkpointer=checkpointer,
    )
    print("── Step 1: Initial card ──")
    print(r1["message"])

    r2 = await handle_message(
        user_id=USER_ID,
        user_input="actually 2 bananas",
        thread_id=THREAD_ID,
        checkpointer=checkpointer,
    )
    print("\n── Step 2: After correction ──")
    print(r2["message"])

    r3 = await handle_message(
        user_id=USER_ID,
        user_input="perfect",
        thread_id=THREAD_ID,
        checkpointer=checkpointer,
    )
    print("\n── Step 3: Logged ──")
    print(r3["message"])

---
## Cell 9 — Classify Intent
Validates the keyword classifier — no LLM, no network.  
**Pass criteria:** all rows show ✅.

In [ ]:
from backend.agent.tools.meals import classify_intent

test_cases = [
    # (input,                    expected)
    ("yes",                      "confirm"),
    ("Looks good!",               "confirm"),
    ("log it",                   "confirm"),
    ("perfect",                  "confirm"),
    ("no",                       "reject"),
    ("cancel",                   "reject"),
    ("never mind",               "reject"),
    ("actually 3 eggs",          "correct"),
    ("add a coffee",             "correct"),
    ("swap oat milk for almond", "correct"),
]

all_pass = True
for text, expected in test_cases:
    got = classify_intent(text)
    ok = got == expected
    all_pass = all_pass and ok
    mark = "✅" if ok else "❌"
    print(f"{mark}  {text:35s}  expected={expected:8s}  got={got}")

print()
print("All pass:" if all_pass else "FAILURES — check classify_intent keyword sets")